# Redes neurais: do perceptron ao backpropagation

O código dos slides, na ordem em que aparece na aula. Execute de cima para baixo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

AZUL, CEU, NAVY, LARANJA, CINZA = "#2264AF", "#8FCEF1", "#27316E", "#D9480F", "#8A99A5"


def regiao(ax, f, lim=(-0.5, 1.5)):
    """Pinta de azul claro a região em que f(pontos) >= 0,5."""
    g = np.linspace(lim[0], lim[1], 300)
    G1, G2 = np.meshgrid(g, g)
    S = f(np.column_stack([G1.ravel(), G2.ravel()])).reshape(G1.shape)
    ax.contourf(G1, G2, S, levels=[0.5, 1e9], colors=[CEU], alpha=0.45)
    ax.contour(G1, G2, S, levels=[0.5], colors=[LARANJA], linewidths=2.5)


def pontos(ax, X, y, titulo="", lim=(-0.5, 1.5)):
    """Classe 0 vazada, classe 1 cheia."""
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=170, facecolor="white", edgecolor=NAVY, linewidth=2.5, zorder=3)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=170, color=NAVY, zorder=3)
    ax.set(xlim=lim, ylim=lim, xticks=[0, 1], yticks=[0, 1], title=titulo, aspect="equal")

## O perceptron

### O perceptron traça uma reta

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

fig, ax = plt.subplots(figsize=(5, 5))
regiao(ax, lambda P: (P @ np.array([2, 1]) - 3 >= 0).astype(float))
pontos(ax, X, y_and)
ax.set(xlabel="$x_1$", ylabel="$x_2$")
ax.annotate("$\\hat{y} = 1$", (1.22, 1.3), color=AZUL, fontsize=15)
ax.annotate("$\\hat{y} = 0$", (-0.35, -0.3), color=NAVY, fontsize=15)
plt.show()

### Em código: um exemplo, passo a passo

In [ ]:
def degrau(z):
    return 1 if z >= 0 else 0

w = np.zeros(2)
b = 0.0
eta = 1.0

x_i, y_i = X[0], y_and[0]          # o exemplo (0, 0), resposta 0

z = w @ x_i + b                    # 1. soma ponderada
y_chapeu = degrau(z)               # 2. previsão
erro = y_i - y_chapeu              # 3. erro
w = w + eta * erro * x_i           # 4. correção dos pesos
b = b + eta * erro                 #    e do viés

print("z:", z, "| previsão:", y_chapeu, "| erro:", erro)
print("pesos depois:", w, "| viés depois:", b)

### Em código: a mesma conta, para todo exemplo e toda época

In [ ]:
def treinar_perceptron(X, y, eta=1.0, epocas=8):
    w = np.zeros(X.shape[1])
    b = 0.0
    historico = []
    for epoca in range(1, epocas + 1):
        erros = 0
        for x_i, y_i in zip(X, y):
            erro = y_i - degrau(w @ x_i + b)
            w = w + eta * erro * x_i
            b = b + eta * erro
            erros = erros + abs(erro)
        historico.append({"época": epoca, "erros": erros, "w1": w[0], "w2": w[1], "b": b})
    return pd.DataFrame(historico).set_index("época")

### Seis épocas bastam para a porta E

In [ ]:
hist_and = treinar_perceptron(X, y_and)
hist_and

In [ ]:
fig, eixos = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(8.4, 5.6))
for ax, (epoca, linha) in zip(eixos.ravel(), hist_and.head(6).iterrows()):
    w_ep, b_ep = np.array([linha["w1"], linha["w2"]]), linha["b"]
    regiao(ax, lambda P: (P @ w_ep + b_ep >= 0).astype(float))
    pontos(ax, X, y_and, titulo=f"época {epoca}: {int(linha['erros'])} erro(s)")
fig.tight_layout()
plt.show()

### Três portas lógicas, e uma não aprende

In [ ]:
y_or = np.array([0, 1, 1, 1])
y_xor = np.array([0, 1, 1, 0])
portas = {"E": y_and, "OU": y_or, "OU exclusivo (XOR)": y_xor}

fig, eixos = plt.subplots(1, 3, sharey=True, figsize=(12, 3.9))
for ax, (nome, y_porta) in zip(eixos, portas.items()):
    hist = treinar_perceptron(X, y_porta, epocas=20)
    cor = LARANJA if "XOR" in nome else AZUL
    ax.plot(hist.index, hist["erros"], color=cor, lw=2, marker="o", ms=6)
    ax.set(title=nome, xlabel="época", ylim=(-0.3, 4.3), xticks=[1, 5, 10, 15, 20], yticks=[0, 1, 2, 3, 4])
eixos[0].set_ylabel("erros na época")
fig.tight_layout()
plt.show()

### Nenhuma reta separa o XOR

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(12, 4.2))
regiao(eixos[0], lambda P: (P @ np.array([2, 1]) - 3 >= 0).astype(float))
pontos(eixos[0], X, y_and, "E: uma reta resolve")
regiao(eixos[1], lambda P: (P @ np.array([1, 1]) - 0.5 >= 0).astype(float))
pontos(eixos[1], X, y_or, "OU: uma reta resolve")
pontos(eixos[2], X, y_xor, "XOR: nenhuma reta resolve")
for ax in eixos:
    ax.set(xlabel="$x_1$")
eixos[0].set_ylabel("$x_2$")
fig.tight_layout()
plt.show()

## O neurônio que desce o gradiente

### O degrau não diz para que lado ir

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

zz = np.linspace(-6, 6, 400)
fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.step(zz, (zz >= 0).astype(float), color=CINZA, lw=2, where="post")
ax.plot(zz, sigmoide(zz), color=AZUL, lw=3)
ax.annotate("degrau", (-5.6, 0.07), color=CINZA, fontsize=14)
ax.annotate("sigmoide", (1.6, 0.66), color=AZUL, fontsize=14)
ax.set(xlabel="$z$", ylabel="saída", yticks=[0, 0.5, 1])
plt.show()

### A perda mede quão ruim foi a previsão

In [ ]:
p = np.linspace(0.01, 0.999, 300)
fig, ax = plt.subplots(figsize=(6, 4.8))
ax.plot(p, -np.log(p), color=AZUL, lw=3)
ax.scatter([0.574], [-np.log(0.574)], s=110, color=LARANJA, zorder=3)
ax.annotate("o exemplo:\n$\\hat{y} = 0{,}574$", (0.6, 0.85), color=LARANJA, fontsize=13)
ax.set(xlabel="$\\hat{y}$ (quando a resposta é 1)", ylabel="perda $L = -\\ln \\hat{y}$", ylim=(0, 4.7))
plt.show()

### Gradiente descendente: um passo contra a derivada

In [ ]:
ww = np.linspace(-1.2, 3.2, 300)
custo = lambda w: (w - 1.4) ** 2 * 0.55 + 0.3
passos = [-0.8]
for _ in range(5):
    passos.append(passos[-1] - 0.45 * 1.1 * (passos[-1] - 1.4))
passos = np.array(passos)
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.plot(ww, custo(ww), color=AZUL, lw=3)
ax.plot(passos, custo(passos), color=LARANJA, lw=1.5, marker="o", ms=9)
ax.annotate("começo", (passos[0] + 0.1, custo(passos[0]) + 0.05), color=LARANJA, fontsize=13)
ax.set(xlabel="um peso $w$", ylabel="custo $J$", yticks=[])
plt.show()

### Em código: o mesmo passo

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

w = np.array([0.4, -0.6])
b = 0.1
x = np.array([2.0, 1.0])
y = 1
eta = 0.5

y_chapeu = sigmoide(w @ x + b)             # 1. prever
perda = -np.log(y_chapeu)                  # 2. medir (y = 1)
erro = y_chapeu - y                        # 3. atribuir
grad_w = erro * x
grad_b = erro
w = w - eta * grad_w                       # 4. corrigir
b = b - eta * grad_b

print("previsão:", y_chapeu.round(3), "| perda:", perda.round(3))
print("gradiente dos pesos:", grad_w.round(3), "| do viés:", grad_b.round(3))
print("pesos novos:", w.round(3), "| viés novo:", b.round(3))
print("previsão nova:", sigmoide(w @ x + b).round(3))

### Em código: treinar na porta E, com todos os exemplos

In [ ]:
w = np.zeros(2)
b = 0.0
eta = 1.0
custos = []

for epoca in range(2000):
    y_chapeu = sigmoide(X @ w + b)                 # as 4 previsões de uma vez
    perdas = -(y_and * np.log(y_chapeu) + (1 - y_and) * np.log(1 - y_chapeu))
    erro = y_chapeu - y_and                        # os 4 erros
    grad_w = X.T @ erro / len(X)                   # média de erro · entrada
    grad_b = erro.mean()
    w = w - eta * grad_w
    b = b - eta * grad_b
    custos.append(perdas.mean())

print("pesos:", w.round(2), "| viés:", b.round(2))

### O custo cai, e as previsões viram probabilidades

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(custos, color=AZUL, lw=2.5)
ax.set(xlabel="época", ylabel="custo $J$", ylim=(0, 0.75), xlim=(0, 2000))
plt.show()

In [ ]:
pd.DataFrame(
    {"resposta": y_and,
     "previsão": sigmoide(X @ w + b).round(3)},
    index=["(0, 0)", "(0, 1)", "(1, 0)", "(1, 1)"],
)

### E o XOR? O mesmo treino, agora numa função

In [ ]:
def treinar_neuronio(X, y, eta=1.0, epocas=2000):
    w = np.zeros(X.shape[1])
    b = 0.0
    custos = []
    for epoca in range(epocas):
        y_chapeu = sigmoide(X @ w + b)
        perdas = -(y * np.log(y_chapeu) + (1 - y) * np.log(1 - y_chapeu))
        erro = y_chapeu - y
        w = w - eta * X.T @ erro / len(X)
        b = b - eta * erro.mean()
        custos.append(perdas.mean())
    return w, b, custos

y_or = np.array([0, 1, 1, 1])
y_xor = np.array([0, 1, 1, 0])

### O neurônio sigmoide também não aprende o XOR

In [ ]:
linhas = []
for nome, y_porta in [("E", y_and), ("OU", y_or), ("XOR", y_xor)]:
    w, b, custos = treinar_neuronio(X, y_porta)
    linhas.append({"porta": nome, "custo final": round(custos[-1], 3), "pesos": w.round(2),
                   "previsões para (0,0) (0,1) (1,0) (1,1)": sigmoide(X @ w + b).round(2)})

print("custo de quem responde 0,5 para tudo:", -np.log(0.5).round(3))
pd.DataFrame(linhas).set_index("porta")

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 2.0))
for nome, y_porta, cor in [("E", y_and, AZUL), ("OU", y_or, NAVY), ("XOR", y_xor, LARANJA)]:
    custos_porta = treinar_neuronio(X, y_porta)[2]
    ax.plot(custos_porta, color=cor, lw=2.5)
ax.annotate("XOR", (1500, 0.6), color=LARANJA, fontsize=13)
ax.annotate("E e OU", (300, 0.12), color=AZUL, fontsize=13)
ax.set(xlabel="época", ylabel="custo $J$", xlim=(0, 2000), ylim=(0, 0.8))
plt.show()

### Por que o gradiente não ajuda no XOR

In [ ]:
w = np.zeros(2)
b = 0.0
y_chapeu = sigmoide(X @ w + b)
erro = y_chapeu - y_xor

pd.DataFrame({"XOR": y_xor, "previsão": y_chapeu, "erro": erro,
              "erro · x1": erro * X[:, 0] + 0.0, "erro · x2": erro * X[:, 1] + 0.0},
             index=["(0, 0)", "(0, 1)", "(1, 0)", "(1, 1)"])

In [ ]:
print("gradiente dos pesos:", X.T @ erro / len(X), "| gradiente do viés:", erro.mean())

## A rede multicamada

### Em código: a rede feita à mão acerta os quatro casos

In [ ]:
linhas = []
for x1, x2 in X:
    h1 = degrau(1 * x1 + 1 * x2 - 1)        # OU
    h2 = degrau(-1 * x1 - 1 * x2 + 1)       # NÃO-E
    saida = degrau(1 * h1 + 1 * h2 - 2)     # E
    linhas.append({"x1": x1, "x2": x2, "h1 (OU)": h1, "h2 (NÃO-E)": h2, "saída": saida})

tabela_xor = pd.DataFrame(linhas).set_index(["x1", "x2"])
tabela_xor["XOR"] = y_xor
tabela_xor

### A camada oculta redesenha o problema

In [ ]:
H = tabela_xor[["h1 (OU)", "h2 (NÃO-E)"]].to_numpy()
fig, eixos = plt.subplots(1, 2, figsize=(10.5, 4.6))
pontos(eixos[0], X, y_xor, "o que a rede recebe: $(x_1, x_2)$")
eixos[0].set(xlabel="$x_1$", ylabel="$x_2$")
regiao(eixos[1], lambda P: (P @ np.array([1, 1]) - 1.5 >= 0).astype(float))
pontos(eixos[1], H, y_xor, "o que o neurônio de saída recebe: $(h_1, h_2)$")
eixos[1].set(xlabel="$h_1$", ylabel="$h_2$")
eixos[1].annotate("(0,1) e (1,0)\ncaem no mesmo ponto", (1.0, 1.0), (0.05, 0.45), color=NAVY, fontsize=11,
                  arrowprops=dict(arrowstyle="-", color=NAVY))
fig.tight_layout()
plt.show()

### Em código: a ida, linha por linha

In [ ]:
W1 = np.array([[0.1, 0.3], [-0.2, 0.4]])
b1 = np.array([0.1, -0.1])
W2 = np.array([[0.5, -0.3]])
b2 = np.array([0.2])

x = np.array([0.5, 1.0])
y = 1

z1 = W1 @ x + b1            # camada 1: somas
a1 = sigmoide(z1)           #           ativações
z2 = W2 @ a1 + b2           # camada 2: soma
a2 = sigmoide(z2)           #           ativação = previsão
perda = -np.log(a2)

print("z1:", z1, "| a1:", a1.round(3))
print("z2:", z2.round(3), "| previsão:", a2.round(3), "| perda:", perda.round(3))

### Três funções de ativação que você vai encontrar

In [ ]:
zz = np.linspace(-5, 5, 400)
s = sigmoide(zz)
t = np.tanh(zz)
funcoes = [
    ("sigmoide", s, s * (1 - s), (-0.1, 1.1)),
    ("tanh", t, 1 - t**2, (-1.1, 1.1)),
    ("ReLU", np.maximum(0, zz), (zz > 0).astype(float), (-0.3, 5.2)),
]
fig, eixos = plt.subplots(1, 3, figsize=(12.5, 2.9))
for ax, (nome, f, df, ylim) in zip(eixos, funcoes):
    ax.plot(zz, f, color=AZUL, lw=3)
    ax.plot(zz, df, color=LARANJA, lw=2, ls="--")
    ax.set(title=nome, xlabel="$z$", ylim=ylim)
eixos[0].annotate("$g(z)$", (2.6, 0.78), color=AZUL, fontsize=13)
eixos[0].annotate("derivada $g'(z)$", (0.9, 0.3), color=LARANJA, fontsize=13)
fig.tight_layout()
plt.show()

### Mais neurônios ocultos, curvas mais flexíveis

In [ ]:
from sklearn.neural_network import MLPRegressor

rng = np.random.default_rng(3)
x_curva = np.sort(rng.uniform(-3, 3, 120))
y_curva = np.sin(2 * x_curva) + 0.3 * x_curva + rng.normal(0, 0.15, 120)
grade = np.linspace(-3, 3, 300)

fig, eixos = plt.subplots(1, 3, figsize=(12.5, 3.9), sharey=True)
for ax, n in zip(eixos, [1, 3, 20]):
    modelo = MLPRegressor(hidden_layer_sizes=(n,), activation="tanh", solver="lbfgs", max_iter=5000, random_state=1)
    modelo.fit(x_curva.reshape(-1, 1), y_curva)
    ax.scatter(x_curva, y_curva, s=14, color=CINZA, alpha=0.6)
    ax.plot(grade, modelo.predict(grade.reshape(-1, 1)), color=AZUL, lw=3)
    ax.set(title=f"{n} neurônio(s) na camada oculta", xlabel="$x$")
eixos[0].set_ylabel("$y$")
fig.tight_layout()
plt.show()

## A saída e a perda

### Em código: softmax e entropia cruzada

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())        # subtrair o máximo evita números gigantes
    return e / e.sum()

z = np.array([2.0, 1.0, 0.1])
y = np.array([0, 1, 0])            # a classe certa é a segunda

y_chapeu = softmax(z)
perda = -np.sum(y * np.log(y_chapeu))
erro = y_chapeu - y

print("perda:", perda.round(3))
pd.DataFrame({"placar z": z, "probabilidade": y_chapeu.round(3), "resposta y": y, "erro": erro.round(3)},
             index=["gato", "cachorro", "pássaro"])

## Backpropagation

### Em código: a volta, linha por linha

In [ ]:
W1 = np.array([[0.1, 0.3], [-0.2, 0.4]])
b1 = np.array([0.1, -0.1])
W2 = np.array([[0.5, -0.3]])
b2 = np.array([0.2])
x = np.array([0.5, 1.0])
y = 1

In [ ]:
z1 = W1 @ x + b1                         # ida (guardando tudo)
a1 = sigmoide(z1)
z2 = W2 @ a1 + b2
a2 = sigmoide(z2)

delta2 = a2 - y                          # passo 1: erro da saída
grad_W2 = np.outer(delta2, a1)           # passo 2: gradientes da camada 2
grad_b2 = delta2
delta1 = (W2.T @ delta2) * a1 * (1 - a1) # passo 3: erro de volta à camada 1
grad_W1 = np.outer(delta1, x)            # passo 4: gradientes da camada 1
grad_b1 = delta1

print("delta2:", delta2.round(3), "| delta1:", delta1.round(3))
print("grad_W2:", grad_W2.round(3))
print("grad_W1:", grad_W1.round(3).tolist())

### Em código: a correção

In [ ]:
eta = 0.5
W1_novo = W1 - eta * grad_W1
b1_novo = b1 - eta * grad_b1
W2_novo = W2 - eta * grad_W2
b2_novo = b2 - eta * grad_b2

print("W1 novo:", W1_novo.round(3).tolist(), "| b1 novo:", b1_novo.round(3))
print("W2 novo:", W2_novo.round(3).tolist(), "| b2 novo:", b2_novo.round(3))

### Em código: a ida de conferência

In [ ]:
a1_novo = sigmoide(W1_novo @ x + b1_novo)
a2_novo = sigmoide(W2_novo @ a1_novo + b2_novo)

pd.DataFrame(
    {"previsão": [a2[0], a2_novo[0]], "perda": [-np.log(a2[0]), -np.log(a2_novo[0])]},
    index=["antes do passo", "depois do passo"],
).round(3)

### Como saber se a volta está certa: o método ingênuo confere

In [ ]:
def perda_da_rede(W1, b1, W2, b2):
    a2 = sigmoide(W2 @ sigmoide(W1 @ x + b1) + b2)
    return -np.log(a2[0])

eps = 1e-6
perda_original = perda_da_rede(W1, b1, W2, b2)
linhas = []
for j in range(2):
    for k in range(2):
        W1_mexido = W1.copy()
        W1_mexido[j, k] = W1_mexido[j, k] + eps
        numerico = (perda_da_rede(W1_mexido, b1, W2, b2) - perda_original) / eps
        linhas.append({"peso": f"W1[{j},{k}]",
                       "backpropagation": grad_W1[j, k], "mexendo no peso": numerico})

pd.DataFrame(linhas).set_index("peso").round(5)

### Em código: qualquer número de camadas

In [ ]:
def ida(x, W, b):
    a = [x]                                   # a[0] é a entrada
    for W_l, b_l in zip(W, b):
        a.append(sigmoide(W_l @ a[-1] + b_l))
    return a                                  # as ativações de todas as camadas

def volta(y, W, a):
    grad_W = [None] * len(W)
    grad_b = [None] * len(W)
    delta = a[-1] - y                         # erro da saída
    for l in range(len(W) - 1, -1, -1):       # da última camada à primeira
        grad_W[l] = np.outer(delta, a[l])
        grad_b[l] = delta
        if l > 0:                             # erro de volta uma camada
            delta = (W[l].T @ delta) * a[l] * (1 - a[l])
    return grad_W, grad_b

### Em código: a rede aprende o XOR sozinha

In [ ]:
rng = np.random.default_rng(1)
W = [rng.normal(0, 1, (2, 2)), rng.normal(0, 1, (1, 2))]     # rede 2-2-1
b = [np.zeros(2), np.zeros(1)]
eta = 0.5
custos = []

for epoca in range(5000):
    perda_da_epoca = 0.0
    for x_i, y_i in zip(X, y_xor):
        a = ida(x_i, W, b)                                    # ida
        grad_W, grad_b = volta(y_i, W, a)                     # volta
        for l in range(len(W)):                               # correção
            W[l] = W[l] - eta * grad_W[l]
            b[l] = b[l] - eta * grad_b[l]
        y_chapeu = a[-1][0]
        perda_da_epoca += -(y_i * np.log(y_chapeu) + (1 - y_i) * np.log(1 - y_chapeu))
    custos.append(perda_da_epoca / len(X))

### O que nenhum neurônio sozinho fazia

In [ ]:
linhas = []
for x_i, y_i in zip(X, y_xor):
    a = ida(x_i, W, b)
    linhas.append({"x1": x_i[0], "x2": x_i[1], "XOR": y_i, "h1": a[1][0], "h2": a[1][1], "previsão": a[2][0]})
pd.DataFrame(linhas).set_index(["x1", "x2"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 2.5))
ax.plot(custos, color=AZUL, lw=2.5)
ax.set(xlabel="época", ylabel="custo", ylim=(0, None), xlim=(0, 5000))
plt.show()

In [ ]:
def rede_xor(P):
    return np.array([ida(p, W, b)[-1][0] for p in P])

def saida_xor(Hp):
    return sigmoide(Hp @ W[1][0] + b[1][0])

H_aprendido = np.array([ida(x_i, W, b)[1] for x_i in X])
fig, eixos = plt.subplots(1, 2, figsize=(8.6, 4.4))
g = np.linspace(-0.5, 1.5, 70)
G1, G2 = np.meshgrid(g, g)
S = rede_xor(np.column_stack([G1.ravel(), G2.ravel()])).reshape(G1.shape)
eixos[0].contourf(G1, G2, S, levels=[0.5, 2], colors=[CEU], alpha=0.45)
eixos[0].contour(G1, G2, S, levels=[0.5], colors=[LARANJA], linewidths=2.5)
pontos(eixos[0], X, y_xor, "nas entradas: fronteira dobrada")
eixos[0].set(xlabel="$x_1$", ylabel="$x_2$")
regiao(eixos[1], saida_xor, lim=(-0.15, 1.15))
pontos(eixos[1], H_aprendido, y_xor, "na camada oculta: uma reta", lim=(-0.15, 1.15))
eixos[1].set(xlabel="$h_1$", ylabel="$h_2$")
fig.tight_layout()
plt.show()

### O erro encolhe a cada camada que atravessa

In [ ]:
camadas_atras = np.arange(0, 11)
fig, ax = plt.subplots(figsize=(7, 4.3))
ax.semilogy(camadas_atras, 0.25**camadas_atras, color=LARANJA, lw=2.5, marker="o", ms=8)
ax.semilogy(camadas_atras, 1.0**camadas_atras, color=AZUL, lw=2.5, marker="o", ms=8)
ax.annotate("sigmoide: no máximo $0{,}25^n$", (5.4, 4e-3), color=LARANJA, fontsize=13)
ax.annotate("ReLU (lado positivo): $1^n$", (5.4, 2.4), color=AZUL, fontsize=13)
ax.set(xlabel="camadas atravessadas na volta", ylabel="fator sobre o erro (escala log)", ylim=(5e-7, 20))
plt.show()

## Treinar de verdade

### O problema: reconhecer dígitos escritos à mão

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digitos = load_digits()
fig, eixos = plt.subplots(2, 10, figsize=(12.5, 2.9))
for ax, imagem, rotulo in zip(eixos.ravel(), digitos.images, digitos.target):
    ax.imshow(imagem, cmap="Blues")
    ax.set_title(str(rotulo), fontsize=13)
    ax.axis("off")
fig.tight_layout()
plt.show()

### Preparar os dados

In [ ]:
X_dig = digitos.data / 16                    # pixels de 0 a 16 viram 0 a 1
y_dig = digitos.target

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_dig, y_dig, test_size=0.25, random_state=0, stratify=y_dig
)
Y_treino = np.eye(10)[y_treino]              # respostas em one-hot

print("treino:", X_treino.shape, "| teste:", X_teste.shape)
print("resposta da 1ª imagem:", y_treino[0], "| em one-hot:", Y_treino[0])

### As peças da rede 64-32-10

In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def softmax_lote(Z):                              # softmax em cada linha
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

rng = np.random.default_rng(0)
W1 = rng.normal(0, np.sqrt(2 / 64), (32, 64))     # pesos pequenos e aleatórios
b1 = np.zeros(32)
W2 = rng.normal(0, np.sqrt(2 / 32), (10, 32))
b2 = np.zeros(10)

print("W1:", W1.shape, "| b1:", b1.shape, "| W2:", W2.shape, "| b2:", b2.shape)
print("parâmetros:", W1.size + b1.size + W2.size + b2.size)

### Com um lote, cada linha da matriz é um exemplo

In [ ]:
X_lote, Y_lote = X_treino[:32], Y_treino[:32]     # as 32 primeiras imagens

Z1 = X_lote @ W1.T + b1                           # ida, 32 exemplos de uma vez
A1 = relu(Z1)
Z2 = A1 @ W2.T + b2
P = softmax_lote(Z2)

print("formas -> X_lote:", X_lote.shape, "| Z1:", Z1.shape, "| P:", P.shape)
print("probabilidades da 1ª imagem:", P[0].round(2))

### A volta do lote: as mesmas quatro equações

In [ ]:
m = len(X_lote)
custo = -np.sum(Y_lote * np.log(P)) / m

D2 = (P - Y_lote) / m                 # eq. 1: erro da saída (já com a média)
grad_W2 = D2.T @ A1                   # eq. 3
grad_b2 = D2.sum(axis=0)              # eq. 4
D1 = (D2 @ W2) * (Z1 > 0)             # eq. 2: (Z1 > 0) é a derivada da ReLU
grad_W1 = D1.T @ X_lote
grad_b1 = D1.sum(axis=0)

print("custo do lote antes de treinar:", custo.round(3))
print("formas dos gradientes -> W2:", grad_W2.shape, "| W1:", grad_W1.shape)

### O laço de treino completo

In [ ]:
def treinar_digitos(eta, epocas=30, tamanho_lote=32, semente=0):
    rng = np.random.default_rng(semente)
    W1 = rng.normal(0, np.sqrt(2 / 64), (32, 64)); b1 = np.zeros(32)
    W2 = rng.normal(0, np.sqrt(2 / 32), (10, 32)); b2 = np.zeros(10)
    historico = []
    for epoca in range(1, epocas + 1):
        ordem = rng.permutation(len(X_treino))                 # embaralha a cada época
        for inicio in range(0, len(ordem), tamanho_lote):
            lote = ordem[inicio:inicio + tamanho_lote]
            A0, Y = X_treino[lote], Y_treino[lote]
            Z1 = A0 @ W1.T + b1; A1 = relu(Z1)                 # ida
            P = softmax_lote(A1 @ W2.T + b2)
            D2 = (P - Y) / len(lote)                           # volta
            D1 = (D2 @ W2) * (Z1 > 0)
            W2 -= eta * D2.T @ A1; b2 -= eta * D2.sum(axis=0)  # correção
            W1 -= eta * D1.T @ A0; b1 -= eta * D1.sum(axis=0)
        P_treino = softmax_lote(relu(X_treino @ W1.T + b1) @ W2.T + b2)
        historico.append({"época": epoca, "custo": -np.mean(np.log(P_treino[Y_treino == 1] + 1e-12))})
    return (W1, b1, W2, b2), pd.DataFrame(historico).set_index("época")

### A rede escrita à mão acerta quase todos os dígitos de teste

In [ ]:
pesos, hist = treinar_digitos(eta=0.1)
W1, b1, W2, b2 = pesos

def prever(X):
    A1 = relu(X @ W1.T + b1)
    P = softmax_lote(A1 @ W2.T + b2)
    return P.argmax(axis=1)     # classe mais provável

acertos_treino = (prever(X_treino) == y_treino).mean()
acertos_teste = (prever(X_teste) == y_teste).mean()
print("acurácia no treino:", acertos_treino.round(3))
print("acurácia no teste: ", acertos_teste.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.2))
ax.plot(hist.index, hist["custo"], color=AZUL, lw=2.5)
ax.set(xlabel="época", ylabel="custo no treino", ylim=(0, None), xlim=(1, 30))
plt.show()

### Onde ela erra

In [ ]:
previsto = prever(X_teste)
errados = np.flatnonzero(previsto != y_teste)[:10]
fig, eixos = plt.subplots(1, len(errados), figsize=(12.5, 1.9))
for ax, i in zip(eixos, errados):
    ax.imshow(X_teste[i].reshape(8, 8), cmap="Blues")
    ax.set_title(f"era {y_teste[i]}\ndisse {previsto[i]}", fontsize=12, color=LARANJA)
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
print("imagens de teste:", len(y_teste), "| erradas:", int((previsto != y_teste).sum()))

### A taxa de aprendizado decide se o treino funciona

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.1))
estilos = {0.001: (CINZA, "$\\eta = 0{,}001$: passos pequenos demais"),
           0.1: (AZUL, "$\\eta = 0{,}1$: desce bem"),
           3.0: (LARANJA, "$\\eta = 3$: passos grandes demais, não desce")}
for eta_teste, (cor, rotulo) in estilos.items():
    _, h = treinar_digitos(eta=eta_teste)
    ax.plot(h.index, h["custo"], color=cor, lw=2.5)
    ax.annotate(rotulo, (30.4, h["custo"].iloc[-1]), color=cor, fontsize=12, va="center", annotation_clip=False)
ax.set(xlabel="época", ylabel="custo no treino", xlim=(1, 30), ylim=(0, 2.7))
fig.subplots_adjust(right=0.62)
plt.show()

### A mesma rede em cinco linhas de scikit-learn

In [ ]:
from sklearn.neural_network import MLPClassifier

modelo = MLPClassifier(hidden_layer_sizes=(32,), activation="relu", solver="sgd",
                       learning_rate_init=0.1, batch_size=32, max_iter=30, random_state=0)
modelo.fit(X_treino, y_treino)

pd.DataFrame(
    {"acurácia no teste": [acertos_teste, modelo.score(X_teste, y_teste)]},
    index=["nossa rede, em numpy", "MLPClassifier"],
).round(3)